Feature	FAISS	ChromaDB
Flat index	✅	❌
IVF	✅	❌
HNSW	✅	✅
PQ	✅	❌
Manual index selection	✅	❌
Metadata filtering	Limited	✅
Persistence	Manual	Built-in
LangChain integration	✅	✅

Final understanding FAISS = vector index/search engine

You manually manage:

index type
dimension
metric
docstore
ID mapping
persistence
Chroma = vector database It manages:

vectors
documents
metadata
IDs
collections
index
persistence

FAISS is a low-level, high-performance library for dense-vector similarity search and clustering. It gives developers direct control over index structures such as Flat, IVF, HNSW and product-quantized indexes, and it offers strong CPU and GPU capabilities. However, FAISS is not a complete vector database: document storage, metadata management, filtering, CRUD APIs, collections, persistence orchestration and server infrastructure generally need to be handled separately.

Chroma is a retrieval database/search infrastructure designed for AI applications. It stores embeddings together with documents, metadata and IDs, and provides collections, persistence, metadata filtering, full-text and sparse retrieval, CRUD operations and client-server or hosted deployment options. In current Chroma, single-node vector search uses HNSW, while its broader schema and cloud architecture also support other retrieval indexes such as SPANN and sparse/full-text indexes.

Therefore, FAISS is preferable when low-level index control, custom ANN algorithms, compression or GPU optimization is the priority. Chroma is preferable when building a complete RAG application that needs database-style storage, filtering, updates and operational simplicity.

Feature	FAISS	Chroma
Store vectors	✅	✅
Store documents	❌	✅
Store metadata	❌	✅
Collections	❌	✅
CRUD	Limited	✅
Filtering	❌ (native)	✅
Persistence	Basic index save/load	✅
Client APIs	❌	✅
Server mode	❌	✅

LangChain FAISS VectorStore
│
├── FAISS index
│     └── Numerical embedding vectors
│
├── InMemoryDocstore
│     └── LangChain Document objects
│         ├── page_content
│         └── metadata
│
└── index_to_docstore_id
      └── FAISS position ko Document ID se map karta hai

Component	LangChain + FAISS	Chroma
Embeddings	Native FAISS index	Chroma vector index
Documents	LangChain Docstore	Chroma collection
Metadata	LangChain Document.metadata	Chroma collection record
Mapping	index_to_docstore_id	Internally managed
Save	FAISS file + pickle	Database persistence
Metadata filters	Wrapper/application handling	Native database filtering
Collections	Not native to FAISS	Native
CRUD	Wrapper/index-dependent	Native record operations
Server/cloud	Separate system required	Supported database architecture 


In [12]:
from dotenv import load_dotenv
import os

# from langchain_google_genai import (
#     GoogleGenerativeAIEmbeddings,
#     ChatGoogleGenerativeAI
# )

# from langchain_community.document_loaders import PyPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_chroma import Chroma


# from langchain_huggingface import HuggingFaceEmbeddings

# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnablePassthrough
# from langchain_core.output_parsers import StrOutputParser

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [4]:
#os.environ["rag-practice-key"] = os.getenv("rag-practice-key")
#from langchain_openai import OpenAIEmbeddings
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")



In [5]:
# Load PDF using PyPDFLoader
file_path ="E:\workspace\Database\chromadb\data\llama2-research-paper.pdf"

loader = PyPDFLoader(file_path)
pages = loader.load()

print("Total pages:", len(pages))

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Total pages: 77


In [6]:
# Chunking using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 175


In [7]:
# Store the chunking data into vector db after embedding
# vector_store= Chroma(
#     collection_name="llama2_collection",
#     embedding_function=embeddings,
#     persist_directory="./chroma_db_llama2",
#     collection_metadata={"hnsw:space": "cosine"}    
# )

In [8]:
# Just testng

{
    "id": "doc-101",
    "embedding": [0.12, -0.45, 0.78, ...],
    "document": "Employees receive 20 days of annual leave.",
    "metadata": {
        "source": "hr_policy.pdf",
        "page": 5,
        "department": "HR"
    }
}

{'id': 'doc-101',
 'embedding': [0.12, -0.45, 0.78, Ellipsis],
 'document': 'Employees receive 20 days of annual leave.',
 'metadata': {'source': 'hr_policy.pdf', 'page': 5, 'department': 'HR'}}

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(type(embeddings))
print(hasattr(embeddings, "embed_documents"))


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True


In [23]:
vector_store=Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2"
)

In [24]:
document_ids = vector_store.add_documents(
    documents=chunks
)

In [34]:

print("Documents added:", len(document_ids))

Documents added: 175


In [35]:
print(
    "Total documents stored:",
    vector_store._collection.count()
)


Total documents stored: 175


In [27]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3
    }
)

In [28]:
query = "What is the architecture of Llama 2?"
retrieved_documents = retriever.invoke(query)
for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)


--- Retrieved document 1 ---
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delayi
Metadata: {'trapped': '/False', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'producer': 'pdfTeX-1.40.25', 'page': 3, 'author': '', 'creator': 'LaTeX with hyperref', 'total_pages': 77, 'source': 'E:\\workspace\\Database\\chromadb\\data\\llama2-research-paper.pdf', 'page_label': '4', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': '', 'subject': '', 'creationdate': '2023-07-20T00:30:36+

In [29]:
prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

In [30]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)


In [31]:
def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )


In [32]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [33]:
answer=rag_chain.invoke("what is the architecture of Llama 2?")
print("\nFinal answer:\n")
print(answer)

e:\workspace\myenv10pm\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



Final answer:

Based on the provided context, the architecture of Llama 2 is defined as follows:

* **Model Type:** It is an auto-regressive language model that uses an optimized transformer architecture.
* **Context Length:** It expands the context window from 2048 tokens (in Llama 1) to 4096 tokens.
* **Attention Mechanism:** It utilizes Grouped-Query Attention (GQA) where key and value projections can be shared across multiple heads to manage memory costs associated with the KV cache size.
* **Tuned Versions:** The tuned versions incorporate supervised fine-tuning (SFT) and reinforcement learning with human feedback (RLHF) to align with human preferences for helpfulness and safety.
